In [6]:
import ee
import calendar
import json

def get_era5_temperature_quindecimal_profile(lat, lon, start_year=2016, end_year=2026):
    point = ee.Geometry.Point([lon, lat])
    
    era5_coll = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
        .select('temperature_2m') \
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
    
    profile_data = {}
    
    for year in range(start_year, end_year + 1):
        year_data = {}
        for month in range(1, 13):
            month_str = str(month).zfill(2)
            _, last_day = calendar.monthrange(year, month)
            
            # --- QUINCENA 1: Del 1 al 15 ---
            q1_start = f'{year}-{month_str}-01'
            q1_end = f'{year}-{month_str}-15'
            img_q1 = era5_coll.filterDate(q1_start, q1_end)
            
            # Combinamos los reductores de manera nativa
            combined_reducer = ee.Reducer.mean().combine(
                reducer2=ee.Reducer.stdDev(), sharedInputs=True
            ).combine(
                reducer2=ee.Reducer.variance(), sharedInputs=True
            )
            
            stats_q1 = img_q1.reduce(combined_reducer)
            val_q1 = stats_q1.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=9000,
                maxPixels=1e9
            ).getInfo()
            
            # --- QUINCENA 2: Del 16 al último día ---
            q2_start = f'{year}-{month_str}-16'
            if month == 12:
                q2_end = f'{year+1}-01-01'
            else:
                next_month_str = str(month + 1).zfill(2)
                q2_end = f'{year}-{next_month_str}-01'
                
            img_q2 = era5_coll.filterDate(q2_start, q2_end)
            stats_q2 = img_q2.reduce(combined_reducer)
            val_q2 = stats_q2.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=9000,
                maxPixels=1e9
            ).getInfo()
            
            # Función de procesamiento inteligente para capturar las bandas combinadas de GEE
            def parse_stats(v):
                if not v:
                    return {'media_C': None, 'std_C': None, 'var_C': None}
                
                # GEE asigna nombres como: temperature_2m_mean, temperature_2m_stdDev, temperature_2m_variance
                mean_k = v.get('temperature_2m_mean')
                std_k = v.get('temperature_2m_stdDev')
                var_k = v.get('temperature_2m_variance')
                
                return {
                    'media_C': round(mean_k - 273.15, 2) if mean_k is not None else None,
                    'std_C': round(std_k, 2) if std_k is not None else None,         # Magnitud pura en grados
                    'var_C': round(var_k, 2) if var_k is not None else None          # Varianza pura
                }

            year_data[f'mes_{month_str}_q1'] = parse_stats(val_q1)
            year_data[f'mes_{month_str}_q2'] = parse_stats(val_q2)
            
        profile_data[str(year)] = year_data
        
    return profile_data

# ==========================================
# PRUEBA DEL SCRIPT
# ==========================================
if __name__ == '__main__':
    lat_test, lon_test = 7.300921, -73.009794  # Finca Matanza (Santander, Colombia)
    print(f"📥 Extrayendo perfil térmico quincenal completo para: Lat {lat_test}, Lon {lon_test}...")
    
    resultado_temperatura = get_era5_temperature_quindecimal_profile(lat_test, lon_test, start_year=2020, end_year=2026)
    
    print("\n✅ Extracción exitosa con Media, Desviación Estándar y Varianza (Año 2016):")
    print(json.dumps(resultado_temperatura.get('2016', {}), indent=4))

📥 Extrayendo perfil térmico quincenal completo para: Lat 7.300921, Lon -73.009794...

✅ Extracción exitosa con Media, Desviación Estándar y Varianza (Año 2016):
{}


In [ ]:
import ee
import calendar
import pandas as pd
import json
ee.Initialize()

def get_era5_temperature_quindecimal_profile(lat, lon, start_year=2016, end_year=2026):
    point = ee.Geometry.Point([lon, lat])
    
    era5_coll = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
        .select('temperature_2m') \
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
    
    profile_data = {}
    
    for year in range(start_year, end_year + 1):
        year_data = {}
        for month in range(1, 13):
            month_str = str(month).zfill(2)
            _, last_day = calendar.monthrange(year, month)
            
            # --- QUINCENA 1: Del 1 al 15 ---
            q1_start = f'{year}-{month_str}-01'
            q1_end = f'{year}-{month_str}-15'
            img_q1 = era5_coll.filterDate(q1_start, q1_end)
            
            combined_reducer = ee.Reducer.mean().combine(
                reducer2=ee.Reducer.stdDev(), sharedInputs=True
            ).combine(
                reducer2=ee.Reducer.variance(), sharedInputs=True
            )
            
            stats_q1 = img_q1.reduce(combined_reducer)
            val_q1 = stats_q1.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=9000,
                maxPixels=1e9
            ).getInfo()
            
            # --- QUINCENA 2: Del 16 al último día ---
            q2_start = f'{year}-{month_str}-16'
            if month == 12:
                q2_end = f'{year+1}-01-01'
            else:
                next_month_str = str(month + 1).zfill(2)
                q2_end = f'{year}-{next_month_str}-01'
                
            img_q2 = era5_coll.filterDate(q2_start, q2_end)
            stats_q2 = img_q2.reduce(combined_reducer)
            val_q2 = stats_q2.reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=9000,
                maxPixels=1e9
            ).getInfo()
            
            def parse_stats(v):
                if not v:
                    return {'media_C': None, 'std_C': None, 'var_C': None}
                
                mean_k = v.get('temperature_2m_mean')
                std_k = v.get('temperature_2m_stdDev')
                var_k = v.get('temperature_2m_variance')
                
                return {
                    'media_C': round(mean_k - 273.15, 2) if mean_k is not None else None,
                    'std_C': round(std_k, 2) if std_k is not None else None,
                    'var_C': round(var_k, 2) if var_k is not None else None
                }

            year_data[f'M{month_str}_Q1'] = parse_stats(val_q1)
            year_data[f'M{month_str}_Q2'] = parse_stats(val_q2)
            
        profile_data[str(year)] = year_data
        
    return profile_data

def convert_profile_to_table(profile_dict, year):
    """
    Convierte el diccionario de un año específico en un DataFrame de Pandas
    donde las columnas son las quincenas y las filas son las métricas (media_C, std_C, var_C).
    """
    if str(year) not in profile_dict:
        raise ValueError(f"El año {year} no se encuentra en el diccionario.")
    
    year_data = profile_dict[str(year)]
    
    # Construir el DataFrame transpuesto: Índices = métricas, Columnas = quincenas
    df = pd.DataFrame(year_data)
    
    return df

# ==========================================
# PRUEBA DEL SCRIPT Y VISUALIZACIÓN EN TABLA
# ==========================================
if __name__ == '__main__':
    lat_test, lon_test = 7.300921, -73.009794  # Finca Matanza (Santander, Colombia)
    print(f"📥 Extrayendo perfil térmico quincenal en formato tabular para: Lat {lat_test}, Lon {lon_test}...")
    
    # Extraemos por ejemplo para el año 2016
    resultado_temperatura = get_era5_temperature_quindecimal_profile(lat_test, lon_test, start_year=2021, end_year=2021)
    
    # Generamos la tabla para el año 2016
    df_2016 = convert_profile_to_table(resultado_temperatura, 2021)
    
    print("\n✅ Tabla de Comportamiento Térmico Quincenal (Año 2016):")
    print(df_2016.to_string())

📥 Extrayendo perfil térmico quincenal en formato tabular para: Lat 7.300921, Lon -73.009794...

✅ Tabla de Comportamiento Térmico Quincenal (Año 2016):
         M01_Q1  M01_Q2  M02_Q1  M02_Q2  M03_Q1  M03_Q2  M04_Q1  M04_Q2  M05_Q1  M05_Q2  M06_Q1  M06_Q2  M07_Q1  M07_Q2  M08_Q1  M08_Q2  M09_Q1  M09_Q2  M10_Q1  M10_Q2  M11_Q1  M11_Q2  M12_Q1  M12_Q2
media_C   14.82   15.38   15.88   15.44   15.08   14.94   15.35   15.44   15.14   15.64   15.17   14.93   15.21   15.47   15.36   15.22   15.07   15.19   15.44   15.24   15.06   14.87   15.33   15.29
std_C      0.48    0.30    0.61    0.40    0.50    0.45    0.35    0.41    0.73    0.44    0.44    0.26    0.39    0.53    0.34    0.36    0.36    0.39    0.39    0.32    0.24    0.41    0.45    0.22
var_C      0.23    0.09    0.37    0.16    0.25    0.20    0.12    0.17    0.53    0.19    0.19    0.07    0.15    0.28    0.11    0.13    0.13    0.15    0.15    0.10    0.06    0.17    0.20    0.05


In [9]:
import ee
import calendar
import pandas as pd
import json

# 1. Inicializar Earth Engine
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

def generate_era5_temperature_csv(lat=7.300921, lon=-73.009794, start_year=2016, end_year=2026, filename="temperatura_quincenal_10_anos.csv"):
    """
    Extrae la serie de temperatura quincenal (media, desviación estándar y varianza) 
    de los últimos 10 años utilizando ERA5-Land Daily Aggregates y exporta un CSV
    con la estructura: Fechas por Fila y Variables por Columna.
    """
    point = ee.Geometry.Point([lon, lat])
    
    # Colección de reanálisis ERA5-Land Daily Aggregates (Temperatura a 2 metros en Kelvin)
    era5_coll = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR') \
        .select('temperature_2m') \
        .filterDate(f'{start_year}-01-01', f'{end_year}-12-31')
    
    # Reductor combinado para eficiencia en GEE
    combined_reducer = ee.Reducer.mean().combine(
        reducer2=ee.Reducer.stdDev(), sharedInputs=True
    ).combine(
        reducer2=ee.Reducer.variance(), sharedInputs=True
    )

    all_rows = []
    
    print(f"📥 Procesando datos desde {start_year} hasta {end_year} para lat: {lat}, lon: {lon}...")

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            month_str = str(month).zfill(2)
            _, last_day = calendar.monthrange(year, month)
            
            # --- QUINCENA 1: Del 1 al 15 ---
            q1_start = f'{year}-{month_str}-01'
            q1_end = f'{year}-{month_str}-15'
            img_q1 = era5_coll.filterDate(q1_start, q1_end)
            
            val_q1 = img_q1.reduce(combined_reducer).reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=9000,
                maxPixels=1e9
            ).getInfo()
            
            # --- QUINCENA 2: Del 16 al último día del mes ---
            q2_start = f'{year}-{month_str}-16'
            if month == 12:
                q2_end = f'{year+1}-01-01'
            else:
                next_month_str = str(month + 1).zfill(2)
                q2_end = f'{year}-{next_month_str}-01'
                
            img_q2 = era5_coll.filterDate(q2_start, q2_end)
            
            val_q2 = img_q2.reduce(combined_reducer).reduceRegion(
                reducer=ee.Reducer.first(),
                geometry=point,
                scale=9000,
                maxPixels=1e9
            ).getInfo()
            
            # Procesamiento de unidades (K a °C solo para la media; std y var en magnitud pura)
            def parse_stats(v):
                if not v:
                    return {'media_C': None, 'std_C': None, 'var_C': None}
                
                mean_k = v.get('temperature_2m_mean')
                std_k = v.get('temperature_2m_stdDev')
                var_k = v.get('temperature_2m_variance')
                
                return {
                    'media_C': round(mean_k - 273.15, 2) if mean_k is not None else None,
                    'std_C': round(std_k, 2) if std_k is not None else None,
                    'var_C': round(var_k, 2) if var_k is not None else None
                }
            
            res_q1 = parse_stats(val_q1)
            res_q2 = parse_stats(val_q2)
            
            # Agregar registros organizados por fila (Quincena)
            all_rows.append({
                'fecha_quincena': f'{year}-M{month_str}_Q1',
                'lat_test': lat,
                'lon_test': lon,
                **res_q1
            })
            all_rows.append({
                'fecha_quincena': f'{year}-M{month_str}_Q2',
                'lat_test': lat,
                'lon_test': lon,
                **res_q2
            })
            
    # Crear DataFrame de Pandas y exportar a CSV
    df_final = pd.DataFrame(all_rows)
    df_final.to_csv(filename, index=False)
    
    print(f"\n✅ ¡Archivo generado con éxito! Guardado como: {filename}")
    print(f"📊 Total de registros quincenales procesados: {len(df_final)}")
    return filename, df_final

# ==========================================
# EJECUCIÓN PRINCIPAL DEL SCRIPT
# ==========================================
if __name__ == '__main__':
    # Coordenadas de prueba solicitadas
    LAT_TEST = 7.300921
    LON_TEST = -73.009794
    START = 2016
    END = 2026
    
    archivo_salida, df_resultado = generate_era5_temperature_csv(
        lat=LAT_TEST, 
        lon=LON_TEST, 
        start_year=START, 
        end_year=END
    )
    
    # Mostrar una muestra en consola
    print("\nMuestra de las primeras 5 filas del CSV:")
    print(df_resultado.head(5).to_string())

📥 Procesando datos desde 2016 hasta 2026 para lat: 7.300921, lon: -73.009794...

✅ ¡Archivo generado con éxito! Guardado como: temperatura_quincenal_10_anos.csv
📊 Total de registros quincenales procesados: 264

Muestra de las primeras 5 filas del CSV:
  fecha_quincena  lat_test   lon_test  media_C  std_C  var_C
0    2016-M01_Q1  7.300921 -73.009794    16.36   0.25   0.06
1    2016-M01_Q2  7.300921 -73.009794    16.63   0.32   0.10
2    2016-M02_Q1  7.300921 -73.009794    16.82   0.24   0.06
3    2016-M02_Q2  7.300921 -73.009794    17.24   0.44   0.19
4    2016-M03_Q1  7.300921 -73.009794    17.22   0.32   0.10


In [ ]:
"""
Extrae la serie de temperatura quincenal (media, desviación estándar y
varianza TEMPORAL, es decir entre los días dentro de cada quincena -
no confundir con el std ESPACIAL que calculamos en terrain_profile.py)
para un punto, usando ERA5-Land Daily Aggregates.

CÓMO LEER EL RESULTADO
-----------------------
media_C   Temperatura media de esa quincena, en °C.
std_C     Cuánto varió la temperatura día a día dentro de esa quincena.
          Alto = quincena con días muy dispares (ej. mezcla de días
          frescos y calurosos); bajo = temperatura estable en el período.
var_C     La misma variabilidad, en varianza (std al cuadrado).
"""
import calendar
from datetime import date, datetime, timedelta
from pathlib import Path
import pandas as pd
import ee
ee.Initialize()


def build_biweekly_periods(start_date, end_date):
    """
    Genera la lista de periodos quincenales (1-15, 16-fin de mes) entre
    start_date y end_date, en Python puro (sin llamadas a GEE todavía).
    Solo incluye quincenas ya completadas (period_end <= end_date + 1 día),
    para no meter quincenas parciales/futuras con datos incompletos.
    """
    periods = []
    current = date(start_date.year, start_date.month, 1)

    while current <= end_date:
        year, month = current.year, current.month

        q1_start = date(year, month, 1)
        q1_end = date(year, month, 16)  # exclusivo en filterDate -> cubre días 1-15

        q2_start = date(year, month, 16)
        if month == 12:
            q2_end = date(year + 1, 1, 1)
        else:
            q2_end = date(year, month + 1, 1)

        # Solo agregamos la quincena si ya terminó completamente
        if q1_end <= end_date + timedelta(days=1):
            periods.append((f'{year}-{month:02d}_Q1', q1_start, q1_end))
        if q2_end <= end_date + timedelta(days=1):
            periods.append((f'{year}-{month:02d}_Q2', q2_start, q2_end))

        current = date(year + 1, 1, 1) if month == 12 else date(year, month + 1, 1)

    return periods


def get_temperature_biweekly(lat, lon, start_date="2016-01-01", end_date=None):
    """
    lat, lon: coordenadas del punto
    start_date: fecha de inicio (str "YYYY-MM-DD")
    end_date: fecha de fin (str "YYYY-MM-DD"); None = hoy.
              OJO: ERA5-Land tiene algunos días de latencia en su
              publicación -> las quincenas más recientes podrían no
              estar disponibles aún aunque "ya pasaron" en el calendario.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date() if end_date else date.today()

    periods = build_biweekly_periods(start, end)
    print(f"[DEBUG] {len(periods)} quincenas a procesar, desde {start} hasta {end}")

    point = ee.Geometry.Point([lon, lat])

    era5_coll = (
        ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
        .select('temperature_2m')
        .filterDate(str(start), str(end + timedelta(days=1)))
    )

    combined_reducer = (
        ee.Reducer.mean()
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.variance(), sharedInputs=True)
    )

    # Armamos la lista de periodos como ee.List de diccionarios, para
    # procesarlos TODOS en el servidor con .map() -> una sola llamada
    # de red al final, en vez de una por quincena (240 antes).
    ee_periods = ee.List([
        {'label': label, 'start': str(p_start), 'end': str(p_end)}
        for label, p_start, p_end in periods
    ])

    def compute_period(period):
        period = ee.Dictionary(period)
        p_start = ee.Date(period.get('start'))
        p_end = ee.Date(period.get('end'))

        filtered = era5_coll.filterDate(p_start, p_end)
        stats = filtered.reduce(combined_reducer).reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=point,
            scale=9000,
            maxPixels=1e9
        )

        return ee.Feature(
            None,
            stats
            .set('label', period.get('label'))
            .set('periodo_inicio', p_start.format('YYYY-MM-dd'))
            .set('periodo_fin', p_end.advance(-1, 'day').format('YYYY-MM-dd'))  # último día incluido
        )

    features = ee.FeatureCollection(ee_periods.map(compute_period))
    result = features.getInfo()  # única llamada de red para TODOS los periodos

    rows = []
    for f in result['features']:
        props = f['properties']
        mean_k = props.get('temperature_2m_mean')
        std_k = props.get('temperature_2m_stdDev')
        var_k = props.get('temperature_2m_variance')

        rows.append({
            'periodo_inicio': props.get('periodo_inicio'),
            'periodo_fin': props.get('periodo_fin'),
            'label': props.get('label'),
            # 'lat': lat,
            # 'lon': lon,
            'media_C': round(mean_k - 273.15, 2) if mean_k is not None else None,
            'std_C': round(std_k, 2) if std_k is not None else None,
            'var_C': round(var_k, 2) if var_k is not None else None,
        })

    df = pd.DataFrame(rows)
    df['periodo_inicio'] = pd.to_datetime(df['periodo_inicio'])
    df = df.sort_values('periodo_inicio').reset_index(drop=True)
    return df


def save_temperature_profile(df, out_prefix="temperature_biweekly", output_dir="../databases"):
    """
    Guarda la serie de temperatura con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv (mismo patrón que terrain_profile/soil_profile)
    """
    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path


if __name__ == "__main__":
    
    LAT = -24.8660
    LON = 152.3489
    
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868

    df = get_temperature_biweekly(LAT, LON, start_date="2016-01-01")
    out_path = save_temperature_profile(df)
    print(df.head(10))

[DEBUG] 254 quincenas a procesar, desde 2016-01-01 hasta 2026-08-06
CSV guardado en ../databases/temperature_biweekly-v260806170042.csv (254x8)
  periodo_inicio periodo_fin       label     lat       lon  media_C  std_C  \
0     2016-01-01  2016-01-15  2016-01_Q1 -24.866  152.3489    24.70   1.05   
1     2016-01-16  2016-01-31  2016-01_Q2 -24.866  152.3489    25.57   1.36   
2     2016-02-01  2016-02-15  2016-02_Q1 -24.866  152.3489    25.27   1.55   
3     2016-02-16  2016-02-29  2016-02_Q2 -24.866  152.3489    25.74   0.83   
4     2016-03-01  2016-03-15  2016-03_Q1 -24.866  152.3489    25.10   0.38   
5     2016-03-16  2016-03-31  2016-03_Q2 -24.866  152.3489    24.44   0.78   
6     2016-04-01  2016-04-15  2016-04_Q1 -24.866  152.3489    23.60   0.73   
7     2016-04-16  2016-04-30  2016-04_Q2 -24.866  152.3489    22.71   0.43   
8     2016-05-01  2016-05-15  2016-05_Q1 -24.866  152.3489    22.04   1.50   
9     2016-05-16  2016-05-31  2016-05_Q2 -24.866  152.3489    21.08   1.50  